# EDA Marketing data project

## Phase 1 : Understanding

In [2]:
import pandas as pd
import numpy as np

In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_csv('marketing_data.csv')

print("=== Shape ===")
print(df.shape)
print()

print("=== Dtypes ===")
print(df.dtypes)
print()

print("=== Info ===")
df.info()
print()

print("=== Describe (numeric) ===")
print(df.describe())
print()

print("=== Describe (categorical) ===")
print(df.describe(include='object'))
print()

print("=== Sample rows ===")
print(df.sample(10, random_state=1))

=== Shape ===
(2240, 28)

=== Dtypes ===
ID                       int64
Year_Birth               int64
Education               object
Marital_Status          object
 Income                float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer             object
Recency                  int64
MntWines                 int64
MntFruits                int64
MntMeatProducts          int64
MntFishProducts          int64
MntSweetProducts         int64
MntGoldProds             int64
NumDealsPurchases        int64
NumWebPurchases          int64
NumCatalogPurchases      int64
NumStorePurchases        int64
NumWebVisitsMonth        int64
AcceptedCmp3             int64
AcceptedCmp4             int64
AcceptedCmp5             int64
AcceptedCmp1             int64
AcceptedCmp2             int64
Response                 int64
Complain                 int64
Country                 object
dtype: object

=== Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries,

## Phase 2 : Cleaning

In [4]:
df.rename(columns={' Income ': 'Income'}, inplace=True)

In [5]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'])

In [6]:
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"Unique IDs: {df['ID'].nunique()} (should be 2240)")

Exact duplicate rows: 0
Unique IDs: 2240 (should be 2240)


In [7]:
income_median = df['Income'].median()
df['Income'] = df['Income'].fillna(income_median)
print(f"Income nulls after fill: {df['Income'].isnull().sum()}")

Income nulls after fill: 0


In [8]:
df.loc[df['Year_Birth'] < 1920, 'Year_Birth'] = np.nan
year_median = df['Year_Birth'].median()
df['Year_Birth'] = df['Year_Birth'].fillna(year_median)
print(f"Year_Birth nulls after fix: {df['Year_Birth'].isnull().sum()}")
print(f"Min Year_Birth now: {df['Year_Birth'].min()}")

Year_Birth nulls after fix: 0
Min Year_Birth now: 1940.0


In [9]:
categorical_cols = ['Education', 'Marital_Status', 'Country']
for col in categorical_cols:
    df[col] = df[col].str.strip().str.title() 

In [10]:
print(df['Marital_Status'].value_counts())
print()
print(df['Country'].value_counts())

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Yolo          2
Absurd        2
Name: count, dtype: int64

Country
Spain           1095
Saudi Arabia     337
Canada           268
Australia        160
India            148
Germany          120
Usa              109
Mexico             3
Name: count, dtype: int64


In [11]:
df['Marital_Status'] = df['Marital_Status'].replace({
    'Alone': 'Single',
    'Yolo': 'Single',
    'Absurd': 'Single',
    'Together': 'Married'
})
print(df['Marital_Status'].value_counts())

Marital_Status
Married     1444
Single       487
Divorced     232
Widow         77
Name: count, dtype: int64


## Phase 3 : Univariate Analysis

In [12]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

numeric_cols = ['Year_Birth', 'Income', 'Recency', 'MntWines', 'MntFruits',
                 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
                 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
                 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']

print("=== Skew per numeric column ===")
print(df[numeric_cols].skew())

=== Skew per numeric column ===
Year_Birth            -0.093704
Income                 6.800947
Recency               -0.001987
MntWines               1.175771
MntFruits              2.102063
MntMeatProducts        2.083233
MntFishProducts        1.919769
MntSweetProducts       2.136081
MntGoldProds           1.886106
NumDealsPurchases      2.418569
NumWebPurchases        1.382794
NumCatalogPurchases    1.880989
NumStorePurchases      0.702237
NumWebVisitsMonth      0.207926
dtype: float64


In [13]:
import os

os.makedirs("plots", exist_ok=True)

for col in numeric_cols:
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f'{col} - Distribution', f'{col} - Boxplot'))
    fig.add_trace(go.Histogram(x=df[col], name=col), row=1, col=1)
    fig.add_trace(go.Box(x=df[col], name=col), row=1, col=2)
    fig.update_layout(height=350, width=800, showlegend=False, title_text=col)
    
    fig.write_image(f"plots/{col}_distribution.png")
    
    fig.show()


In [14]:
categorical_cols = ['Education', 'Marital_Status', 'Country']
print("=== Categorical value counts ===")
for col in categorical_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print(f"nunique: {df[col].nunique()}")
    print()

=== Categorical value counts ===
--- Education ---
Education
Graduation    1127
Phd            486
Master         370
2N Cycle       203
Basic           54
Name: count, dtype: int64
nunique: 5

--- Marital_Status ---
Marital_Status
Married     1444
Single       487
Divorced     232
Widow         77
Name: count, dtype: int64
nunique: 4

--- Country ---
Country
Spain           1095
Saudi Arabia     337
Canada           268
Australia        160
India            148
Germany          120
Usa              109
Mexico             3
Name: count, dtype: int64
nunique: 8



## Phase 4 : Bivariate Analysis

In [15]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px

numeric_cols = ['Year_Birth', 'Income', 'Recency', 'MntWines', 'MntFruits',
                 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
                 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
                 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']
categorical_cols = ['Education', 'Marital_Status', 'Country']

In [20]:
# 1) Numeric x Numeric: correlation matrix + heatmap
corr = df[numeric_cols].corr(method='pearson')
fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                 color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                 title='Pearson correlation - numeric columns')
fig.show()

# flag strongest pairs (excluding self-correlation)
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
print("=== Top 10 strongest numeric-numeric correlations ===")
print(corr_pairs.abs().sort_values(ascending=False).head(10))
print()

print("=== Top 10 strongest NEGATIVE numeric-numeric correlations ===")
print(corr_pairs.sort_values(ascending=True).head(10))
print()

=== Top 10 strongest numeric-numeric correlations ===
MntMeatProducts  NumCatalogPurchases    0.723827
MntWines         NumStorePurchases      0.642100
                 NumCatalogPurchases    0.635226
MntFruits        MntFishProducts        0.594804
Income           NumCatalogPurchases    0.586826
MntFishProducts  MntSweetProducts       0.579870
Income           MntMeatProducts        0.577805
                 MntWines               0.576903
MntMeatProducts  MntFishProducts        0.568402
MntFruits        MntSweetProducts       0.567164
dtype: float64

=== Top 10 strongest NEGATIVE numeric-numeric correlations ===
Income               NumWebVisitsMonth   -0.549785
MntMeatProducts      NumWebVisitsMonth   -0.539470
NumCatalogPurchases  NumWebVisitsMonth   -0.520364
MntFishProducts      NumWebVisitsMonth   -0.446003
NumStorePurchases    NumWebVisitsMonth   -0.428473
MntSweetProducts     NumWebVisitsMonth   -0.423294
MntFruits            NumWebVisitsMonth   -0.418383
MntWines            

In [21]:
# 2) Categorical x Numeric: ANOVA p-value for every combination
print("=== Categorical x Numeric: ANOVA p-values ===")
results = []
for cat in categorical_cols:
    for num in numeric_cols:
        groups = [g[num].dropna().values for _, g in df.groupby(cat)]
        f_stat, p_val = stats.f_oneway(*groups)
        results.append({'categorical': cat, 'numeric': num, 'p_value': p_val, 'significant': p_val < 0.05})
anova_df = pd.DataFrame(results).sort_values('p_value')
print(anova_df.to_string(index=False))
print()


=== Categorical x Numeric: ANOVA p-values ===
   categorical             numeric      p_value  significant
     Education            MntWines 7.104155e-24         True
     Education              Income 1.075202e-22         True
Marital_Status          Year_Birth 2.692906e-21         True
     Education          Year_Birth 1.520895e-19         True
     Education        MntGoldProds 2.526008e-11         True
     Education   NumStorePurchases 3.443579e-10         True
     Education     MntFishProducts 4.858406e-10         True
     Education    MntSweetProducts 1.687573e-09         True
     Education     NumWebPurchases 3.888124e-09         True
     Education           MntFruits 3.568559e-08         True
     Education NumCatalogPurchases 5.803800e-08         True
     Education     MntMeatProducts 1.874092e-06         True
     Education   NumWebVisitsMonth 6.289746e-05         True
Marital_Status     NumWebPurchases 7.894803e-02        False
Marital_Status    MntSweetProducts 7.98

In [22]:
# 3) Categorical x Categorical: chi-square for each pair
print("=== Categorical x Categorical: chi-square p-values ===")
from itertools import combinations
for c1, c2 in combinations(categorical_cols, 2):
    ct = pd.crosstab(df[c1], df[c2])
    chi2, p_val, dof, expected = stats.chi2_contingency(ct)
    print(f"{c1} x {c2}: p = {p_val:.4f} ({'significant' if p_val < 0.05 else 'not significant'})")

=== Categorical x Categorical: chi-square p-values ===
Education x Marital_Status: p = 0.2707 (not significant)
Education x Country: p = 0.3041 (not significant)
Marital_Status x Country: p = 0.0626 (not significant)


## Phase 5 : Outliers Deep Dive

In [23]:
# IQR-based outlier detection لكل عمود، وفحص هل متمركزة في segment معين
outlier_cols = ['Income', 'MntMeatProducts', 'NumCatalogPurchases', 'NumDealsPurchases']

for col in outlier_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"=== {col}: {len(outliers)} outliers (bounds: {lower:.1f} to {upper:.1f}) ===")
    print("By Education:")
    print(outliers['Education'].value_counts(normalize=True).round(3))
    print("By Marital_Status:")
    print(outliers['Marital_Status'].value_counts(normalize=True).round(3))
    print()

# فحص خاص للنقطة الشاذة في Income (القيمة الأعلى)
print("=== Top 3 Income rows ===")
print(df.nlargest(3, 'Income')[['Income', 'Education', 'Marital_Status', 'Country', 'Kidhome', 'Teenhome']])

=== Income: 8 outliers (bounds: -13587.8 to 117416.2) ===
By Education:
Education
Phd           0.500
Graduation    0.375
Master        0.125
Name: proportion, dtype: float64
By Marital_Status:
Marital_Status
Married     0.875
Divorced    0.125
Name: proportion, dtype: float64

=== MntMeatProducts: 175 outliers (bounds: -308.0 to 556.0) ===
By Education:
Education
Graduation    0.514
Phd           0.217
Master        0.194
2N Cycle      0.074
Name: proportion, dtype: float64
By Marital_Status:
Marital_Status
Married     0.606
Single      0.286
Divorced    0.063
Widow       0.046
Name: proportion, dtype: float64

=== NumCatalogPurchases: 23 outliers (bounds: -6.0 to 10.0) ===
By Education:
Education
Graduation    0.652
Phd           0.217
Master        0.130
Name: proportion, dtype: float64
By Marital_Status:
Marital_Status
Married     0.652
Single      0.217
Widow       0.087
Divorced    0.043
Name: proportion, dtype: float64

=== NumDealsPurchases: 86 outliers (bounds: -2.0 to 6.0) ==

In [24]:
df.loc[df['Income'] == 666666, 'Income'] = np.nan
income_median_v2 = df['Income'].median()
df['Income'] = df['Income'].fillna(income_median_v2)
print(f"Income median used: {income_median_v2}")
print(f"Income max now: {df['Income'].max()}")

Income median used: 51381.5
Income max now: 162397.0


## Phase 6 : Feature Engineering

In [25]:
import datetime as dt

# --- Date parts ---
df['Year_Joined'] = df['Dt_Customer'].dt.year
df['Month_Joined'] = df['Dt_Customer'].dt.month
reference_date = df['Dt_Customer'].max()  # آخر تاريخ انضمام في الداتا كمرجع
df['Tenure_Days'] = (reference_date - df['Dt_Customer']).dt.days

In [26]:
# --- Aggregation ---
mnt_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
purchase_cols = ['NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']

df['TotalMnt'] = df[mnt_cols].sum(axis=1)
df['TotalPurchases'] = df[purchase_cols].sum(axis=1)

In [27]:
# --- Flags ---
df['HasKids'] = ((df['Kidhome'] + df['Teenhome']) > 0).astype(int)
campaign_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']
df['TotalCampaignsAccepted'] = df[campaign_cols].sum(axis=1)
df['RespondedAnyCampaign'] = (df['TotalCampaignsAccepted'] > 0).astype(int)

In [28]:
# --- RFM-style ---
df['Recency_Score'] = df['Recency']          # موجود بالفعل
df['Frequency'] = df['TotalPurchases']        # كام مرة اشترى
df['Monetary'] = df['TotalMnt']               # قد إيه صرف

print(df[['Year_Joined', 'Tenure_Days', 'TotalMnt', 'TotalPurchases', 'HasKids',
           'TotalCampaignsAccepted', 'Frequency', 'Monetary']].describe())

       Year_Joined  Tenure_Days     TotalMnt  TotalPurchases      HasKids  TotalCampaignsAccepted    Frequency     Monetary
count  2240.000000  2240.000000  2240.000000     2240.000000  2240.000000             2240.000000  2240.000000  2240.000000
mean   2013.028125   353.582143   605.798214       14.862054     0.715179                0.297768    14.862054   605.798214
std       0.684554   202.122512   602.249288        7.677173     0.451430                0.678381     7.677173   602.249288
min    2012.000000     0.000000     5.000000        0.000000     0.000000                0.000000     0.000000     5.000000
25%    2013.000000   180.750000    68.750000        8.000000     0.000000                0.000000     8.000000    68.750000
50%    2013.000000   355.500000   396.000000       15.000000     1.000000                0.000000    15.000000   396.000000
75%    2013.000000   529.000000  1045.500000       21.000000     1.000000                0.000000    21.000000  1045.500000
max    2

In [29]:
# --- Age groups (quartile-based) ---
current_year = 2014  # آخر سنة في Dt_Customer تقريبًا، نفس المرجع المستخدم في Tenure
df['Age'] = current_year - df['Year_Birth']
df['Age_Group'] = pd.qcut(df['Age'], q=4, labels=['Q1_Youngest', 'Q2', 'Q3', 'Q4_Oldest'])

# --- Income tiers (quartile-based) ---
df['Income_Tier'] = pd.qcut(df['Income'], q=4, labels=['Q1_Lowest', 'Q2', 'Q3', 'Q4_Highest'])

print(df['Age_Group'].value_counts())
print()
print(df['Income_Tier'].value_counts())
print()
print(df.groupby('Age_Group')['Age'].agg(['min', 'max']))
print()
print(df.groupby('Income_Tier')['Income'].agg(['min', 'max']))

Age_Group
Q1_Youngest    606
Q2             561
Q3             557
Q4_Oldest      516
Name: count, dtype: int64

Income_Tier
Q2            573
Q1_Lowest     560
Q4_Highest    560
Q3            547
Name: count, dtype: int64

              min   max
Age_Group              
Q1_Youngest  18.0  37.0
Q2           38.0  44.0
Q3           45.0  55.0
Q4_Oldest    56.0  74.0

                 min       max
Income_Tier                   
Q1_Lowest     1730.0   35523.0
Q2           35544.0   51381.5
Q3           51390.0   68274.0
Q4_Highest   68281.0  162397.0


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13044\874942554.py:13: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13044\874942554.py:15: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [30]:
df.to_csv('marketing_data_cleaned.csv', index=False)

In [31]:
import pandas as pd
import numpy as np

df = pd.read_csv('marketing_data_cleaned.csv')  # الملف الكامل بعد التنظيف

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# --- Q4: Average customer profile ---
print("=== Q4: Customer Profile ===")
print(df[['Age','Income','TotalMnt','TotalPurchases','Tenure_Days']].median())
print(df['Age_Group'].value_counts(normalize=True))
print(df['Income_Tier'].value_counts(normalize=True))
print(df['Education'].value_counts(normalize=True))
print(df['Marital_Status'].value_counts(normalize=True))
print(f"HasKids rate: {df['HasKids'].mean():.2%}")
print()

# --- Q5: Product performance (Pareto) ---
print("=== Q5: Product Performance ===")
mnt_cols = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
product_totals = df[mnt_cols].sum().sort_values(ascending=False)
product_share = (product_totals / product_totals.sum() * 100).round(1)
print(pd.DataFrame({'Total': product_totals, 'Share%': product_share}))
print()

# --- Q6: Channel performance ---
print("=== Q6: Channel Performance ===")
channel_cols = ['NumWebPurchases','NumCatalogPurchases','NumStorePurchases','NumDealsPurchases']
channel_totals = df[channel_cols].sum().sort_values(ascending=False)
channel_share = (channel_totals / channel_totals.sum() * 100).round(1)
print(pd.DataFrame({'Total': channel_totals, 'Share%': channel_share}))
print(f"\nAvg NumWebVisitsMonth: {df['NumWebVisitsMonth'].mean():.2f}")
# نسبة الزيارات اللي بتتحول لشراء ويب (proxy لمعدل تحويل القناة)
print(f"Web conversion proxy (WebPurchases/WebVisits): {(df['NumWebPurchases'].sum()/df['NumWebVisitsMonth'].sum()):.3f}")
print()

# --- Q2: Factors related to web purchases ---
print("=== Q2: Web Purchases Correlations ===")
numeric_cols = ['Income','Age','TotalMnt','Kidhome','Teenhome','NumWebVisitsMonth','Tenure_Days','NumDealsPurchases']
print(df[numeric_cols + ['NumWebPurchases']].corr()['NumWebPurchases'].sort_values(ascending=False))
print()
print("--- ANOVA-style means by Education ---")
print(df.groupby('Education')['NumWebPurchases'].mean().sort_values(ascending=False))
print()

# --- Q3: Most successful campaign ---
print("=== Q3: Campaign Acceptance Rates ===")
cmp_cols = ['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3','AcceptedCmp4','AcceptedCmp5','Response']
acceptance_rates = (df[cmp_cols].mean() * 100).round(2).sort_values(ascending=False)
print(acceptance_rates)
print(f"\nTotalCampaignsAccepted distribution:\n{df['TotalCampaignsAccepted'].value_counts().sort_index()}")
print(f"RespondedAnyCampaign rate: {df['RespondedAnyCampaign'].mean():.2%}")
print()

# --- C3 RFM Segmentation ---
print("=== RFM Segments (customer-level, static) ===")
df['R_score'] = pd.qcut(df['Recency_Score'], 4, labels=[4,3,2,1])  # أقل recency = أفضل
df['F_score'] = pd.qcut(df['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
df['M_score'] = pd.qcut(df['Monetary'].rank(method='first'), 4, labels=[1,2,3,4])
df['RFM_sum'] = df[['R_score','F_score','M_score']].astype(int).sum(axis=1)
print(df.groupby('RFM_sum').size())
print()

# --- B2-lite: Cohort by join year ---
print("=== Cohort-lite by Year_Joined ===")
print(df.groupby('Year_Joined').agg(
    avg_monetary=('Monetary','mean'),
    avg_frequency=('Frequency','mean'),
    response_rate=('RespondedAnyCampaign','mean'),
    n=('ID','count')
))

# --- Pareto check on customers (concentration) ---
print("\n=== Customer Concentration (Pareto) ===")
sorted_mnt = df['TotalMnt'].sort_values(ascending=False)
top20pct_n = int(len(sorted_mnt)*0.2)
top20_share = sorted_mnt.iloc[:top20pct_n].sum() / sorted_mnt.sum()
print(f"Top 20% customers by spend = {top20_share:.1%} of TotalMnt")

=== Q4: Customer Profile ===
Age                  44.0
Income            51381.5
TotalMnt            396.0
TotalPurchases       15.0
Tenure_Days         355.5
dtype: float64
Age_Group
Q1_Youngest    0.270536
Q2             0.250446
Q3             0.248661
Q4_Oldest      0.230357
Name: proportion, dtype: float64
Income_Tier
Q2            0.255804
Q4_Highest    0.250000
Q1_Lowest     0.250000
Q3            0.244196
Name: proportion, dtype: float64
Education
Graduation    0.503125
Phd           0.216964
Master        0.165179
2N Cycle      0.090625
Basic         0.024107
Name: proportion, dtype: float64
Marital_Status
Married     0.644643
Single      0.217411
Divorced    0.103571
Widow       0.034375
Name: proportion, dtype: float64
HasKids rate: 71.52%

=== Q5: Product Performance ===
                   Total  Share%
MntWines          680816    50.2
MntMeatProducts   373968    27.6
MntGoldProds       98609     7.3
MntFishProducts    84057     6.2
MntSweetProducts   60621     4.5
MntFruit

In [32]:
# --- Guardrail check 1: هل علاقة Income/TotalMnt بـ NumWebPurchases بتفضل ثابتة جوه كل Education segment؟ ---
print("=== Guardrail: Web Purchases corr with Income, per Education ===")
for edu in df['Education'].unique():
    sub = df[df['Education'] == edu]
    print(edu, "-> corr Income/WebPurchases:", round(sub['Income'].corr(sub['NumWebPurchases']), 3), f"(n={len(sub)})")
print()

# --- Guardrail check 2: هل Wine لسه مسيطر على المبيعات جوه كل Income_Tier؟ (Simpson's check على Q5) ---
print("=== Guardrail: Product mix share, per Income_Tier ===")
mnt_cols = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
for tier in ['Q1_Lowest','Q2','Q3','Q4_Highest']:
    sub = df[df['Income_Tier'] == tier]
    totals = sub[mnt_cols].sum()
    share = (totals / totals.sum() * 100).round(1)
    print(f"\n{tier} (n={len(sub)}):")
    print(share.sort_values(ascending=False))
print()

# --- Guardrail check 3: هل Store لسه القناة الأكبر جوه كل Age_Group؟ (Simpson's check على Q6) ---
print("=== Guardrail: Channel share, per Age_Group ===")
channel_cols = ['NumWebPurchases','NumCatalogPurchases','NumStorePurchases','NumDealsPurchases']
for ag in ['Q1_Youngest','Q2','Q3','Q4_Oldest']:
    sub = df[df['Age_Group'] == ag]
    totals = sub[channel_cols].sum()
    share = (totals / totals.sum() * 100).round(1)
    print(f"\n{ag} (n={len(sub)}):")
    print(share.sort_values(ascending=False))
print()

# --- Guardrail check 4: هل Response (أنجح حملة) لسه أعلى نسبة جوه كل Income_Tier؟ مش مدفوعة بميل الداتا للـ high income بس ---
print("=== Guardrail: Response rate, per Income_Tier ===")
print(df.groupby('Income_Tier')['Response'].mean().round(3))
print()

# --- تحسين: توصيف الـ RFM segments مش بس أعداد ---
print("=== RFM segment characterization ===")
print(df.groupby('RFM_sum').agg(
    avg_income=('Income','mean'),
    avg_age=('Age','mean'),
    avg_totalmnt=('TotalMnt','mean'),
    haskids_rate=('HasKids','mean'),
    n=('ID','count')
).round(1))

=== Guardrail: Web Purchases corr with Income, per Education ===
Graduation -> corr Income/WebPurchases: 0.464 (n=1127)
Phd -> corr Income/WebPurchases: 0.342 (n=486)
2N Cycle -> corr Income/WebPurchases: 0.555 (n=203)
Master -> corr Income/WebPurchases: 0.402 (n=370)
Basic -> corr Income/WebPurchases: 0.189 (n=54)

=== Guardrail: Product mix share, per Income_Tier ===

Q1_Lowest (n=560):
MntMeatProducts     27.7
MntWines            26.8
MntGoldProds        20.5
MntFishProducts     10.5
MntFruits            7.2
MntSweetProducts     7.2
dtype: float64

Q2 (n=573):
MntWines            52.8
MntMeatProducts     21.9
MntGoldProds        12.0
MntFishProducts      5.9
MntSweetProducts     3.9
MntFruits            3.6
dtype: float64

Q3 (n=547):
MntWines            58.0
MntMeatProducts     21.1
MntGoldProds         7.8
MntFishProducts      5.3
MntFruits            4.0
MntSweetProducts     3.8
dtype: float64

Q4_Highest (n=560):
MntWines            46.9
MntMeatProducts     32.1
MntFishProducts 